# Nearby inputs, nearby answers

If a query cannot change faster than a constant $K$ times the input gap, then nearby states have nearby answers. That is Lipschitz access, Theorem CSQ-T3. If a decoder is everywhere within $\varepsilon$ of the query, two states with the same blurry view have answers at most $2\varepsilon$ apart. That is the collision bound, Theorem CSQ-T2. Neither bound says the two answers are equal.

This notebook uses exact fractions. Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_q_bridge_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_q_bridge_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_q_bridge_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Rational
from coda_cs_q_bridge_core import (
    CofinalShadowTower,
    ProfileFactorizationSolver,
    QueryProbe,
    ShadowNode,
)

## Double a rational, Lipschitz constant $2$

Let $q(t)=2t$. The points $1/2$ and $2/5$ are $1/10$ apart. The answers $1$ and $4/5$ are $1/5$ apart, which is exactly $2$ times the input gap. The bound is sharp on this pair. The two answers are still different.

In [2]:
K = Rational(2)
x = Rational(1, 2)
a = Rational(2, 5)
qx = K * x
qa = K * a
gap = abs(x - a)
qgap = abs(qx - qa)
print("inputs:", x, a, "gap:", gap)
print("answers:", qx, qa, "gap:", qgap)
print("K times input gap:", K * gap)
assert gap == Rational(1, 10)
assert qgap == Rational(1, 5)
assert qgap == K * gap
assert qgap <= K * gap
assert qx != qa

inputs: 1/2 2/5 gap: 1/10
answers: 1 4/5 gap: 1/5
K times input gap: 1/5


## A decoder within $1$ still lets the fibre move by $2$

Look only at evenness of 2 and 4. The identity query sends them to 2 and 4. A decoder that answers 3 on evenness is within 1 of both. Theorem CSQ-T2 says the collision is at most 2. It is exactly 2. The decoder is not an exact factor: 2 is not 4.


In [3]:
domain = [2, 4]
p = lambda n: n % 2
q = lambda n: Rational(n)
g = lambda parity: Rational(3)
eps = max(abs(q(n) - g(p(n))) for n in domain)
collision = abs(q(2) - q(4))
print("decoder error ε:", eps)
print("fibre collision:", collision)
print("2ε:", 2 * eps)
assert eps == Rational(1)
assert collision == Rational(2)
assert collision <= 2 * eps
assert q(2) != q(4)

nodes = [ShadowNode("parity", 1)]
tower = CofinalShadowTower(nodes, {"parity": p}, {})
query = QueryProbe("identity", degree=1, eval_fn=q)
ok, q_bar, err = ProfileFactorizationSolver(tower, query, domain).solve_factorization()
print("exact factor from evenness?", ok)
assert ok is False
assert "Fiber constancy violation" in str(err)

decoder error ε: 1
fibre collision: 2
2ε: 2
exact factor from evenness? False
